# OpenMax vs OsrSAF_TriNet — same closed-set backbone

Both methods sit on top of the same `asymmetric_trinet_seed{seed}_n{n}.pt`
checkpoint, so the comparison is apples-to-apples.

## Steps
1. Mount Drive and set paths.
2. Copy the new files into place (one per src directory + 2 scripts).
3. Train the closed-set `asymmetric_trinet` (skipped if checkpoint exists).
4. Fit OpenMax (MAVs + Weibull tails + Youden threshold).
5. Train OSR-SAF (skipped if checkpoint exists).
6. Evaluate both methods across 13 fixed-SNR datasets and save JSON.

SNR seed mapping used in step 6 (matches the ablation study):
```
s1=410 → +10 dB    s8=264 → -4 dB
s2=118 →  +8 dB    s9=336 → -6 dB
s3=276 →  +6 dB   s10=608 → -8 dB
s4=314 →  +4 dB   s11=530 → -10 dB
s5=152 →  +2 dB   s12=472 → -12 dB
s6=340 →   0 dB   s13=214 → -14 dB
s7=142 →  -2 dB
```

In [10]:
# ── Step 1: Mount Drive and set project root ─────────────────────────────
from google.colab import drive
from pathlib import Path
import sys, os

drive.mount('/content/drive', force_remount=True)

PROJECT_ROOT = Path('/content/drive/Othercomputers/My Laptop/thesis_project')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.chdir(PROJECT_ROOT)
print('Working directory:', os.getcwd())

Mounted at /content/drive
Working directory: /content/drive/Othercomputers/My Laptop/thesis_project


In [13]:
# ── Step 2: Copy the new files into the right places ─────────────────────
# Upload these 5 files via the Files panel (or rsync them into Drive once),
# then this cell will move them into the project tree:
#   - openmax_trinet.py    → python/src/models/
#   - openmax_trainer.py   → python/src/train/
#   - openmax_evaluator.py → python/src/eval/
#   - run_openmax.py       → scripts/
#   - openmax_eval.py      → scripts/
# If a file is already in place this cell prints 'Already in place' and skips.

import shutil

FILE_PLACEMENTS = [
    ('openmax_trinet.py',    PROJECT_ROOT / 'python/src/models/openmax_trinet.py'),
    ('openmax_trainer.py',   PROJECT_ROOT / 'python/src/train/openmax_trainer.py'),
    ('openmax_evaluator.py', PROJECT_ROOT / 'python/src/eval/openmax_evaluator.py'),
    ('run_openmax.py',       PROJECT_ROOT / 'scripts/run_openmax.py'),
    ('openmax_eval.py',      PROJECT_ROOT / 'scripts/openmax_eval.py'),
]

for fname, dst in FILE_PLACEMENTS:
    src = Path('/content') / fname
    dst.parent.mkdir(parents=True, exist_ok=True)
    if src.exists():
        shutil.copy(src, dst)
        print(f'Copied : {dst}')
    elif dst.exists():
        print(f'In place: {dst}')
    else:
        print(f'MISSING: upload {fname} via Files panel, or rsync into Drive.')

# scipy is needed for Weibull tail fitting (only at fit time, not at inference)
try:
    import scipy
    print(f'scipy {scipy.__version__} available')
except ImportError:
    print('Installing scipy...')
    !pip install -q scipy
    import scipy
    print(f'scipy {scipy.__version__} installed')

In place: /content/drive/Othercomputers/My Laptop/thesis_project/python/src/models/openmax_trinet.py
In place: /content/drive/Othercomputers/My Laptop/thesis_project/python/src/train/openmax_trainer.py
In place: /content/drive/Othercomputers/My Laptop/thesis_project/python/src/eval/openmax_evaluator.py
In place: /content/drive/Othercomputers/My Laptop/thesis_project/scripts/run_openmax.py
MISSING: upload openmax_eval.py via Files panel, or rsync into Drive.
scipy 1.16.3 available


In [14]:
# ── Step 3: Train the closed-set backbone (skip if already trained) ──────
# Both OpenMax and OSR-SAF need this checkpoint. If it exists, this cell is
# essentially a no-op.

from python.src.train.model_trainer import train_model
import torch

SEED = 146
N_PER_CLASS = 2500
SPEC_VERSION = 'v2'
EPOCHS_CLOSED = 30

ckpt_closed = PROJECT_ROOT / 'artifacts/checkpoints' / f'asymmetric_trinet_seed{SEED}_n{N_PER_CLASS}.pt'

if ckpt_closed.exists():
    print(f'[SKIP] Closed-set checkpoint exists: {ckpt_closed.name}')
else:
    print(f'Training closed-set asymmetric_trinet | seed={SEED} | n={N_PER_CLASS}')
    print('=' * 78)
    trained = train_model(
        seed=SEED,
        project_root=PROJECT_ROOT,
        model_name='asymmetric_trinet',
        n_per_class=N_PER_CLASS,
        spec_version=SPEC_VERSION,
        n_epochs=EPOCHS_CLOSED,
    )
    del trained
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print('Closed-set training complete.')

AttributeError: partially initialized module 'torch' has no attribute 'types' (most likely due to a circular import)

In [ ]:
# ── Step 4: Fit OpenMax head (MAVs + Weibull + threshold) ────────────────
# This is fast: one pass over the training set + Weibull fit + threshold
# calibration. Saves to artifacts/checkpoints/openmax_trinet_seed{seed}_n{n}.pt
#
# To refit, delete that .pt file first.

%run scripts/run_openmax.py

In [ ]:
# ── Step 5: Train OSR-SAF (skip if already trained) ──────────────────────
# This is your existing training. If the checkpoint exists, skip.

from python.src.train.osr_trainer import train_osr_model
from python.src.train.osr_hparams import OSRHParams
import torch

EPOCHS_OSR = 30
ckpt_osr = PROJECT_ROOT / 'artifacts/checkpoints' / f'osr_saf_trinet_seed{SEED}_n{N_PER_CLASS}.pt'

if ckpt_osr.exists():
    print(f'[SKIP] OSR-SAF checkpoint exists: {ckpt_osr.name}')
else:
    print(f'Training OSR-SAF | seed={SEED} | n={N_PER_CLASS}')
    print('=' * 78)
    hparams = OSRHParams()
    trained = train_osr_model(
        seed=SEED,
        n_per_class=N_PER_CLASS,
        spec_version=SPEC_VERSION,
        project_root=PROJECT_ROOT,
        epochs=EPOCHS_OSR,
        hparams=hparams,
    )
    del trained
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print('OSR-SAF training complete.')

In [ ]:
# ── Step 6: Run the SNR comparison eval and save JSON ────────────────────
# Loops over 13 fixed-SNR datasets × 2 methods and prints comparison tables
# (AUROC, OS-Acc, Known-Acc, Recall, FAR). Saves the full result to:
#   artifacts/logs/openmax/openmax_vs_osr_saf_seed{seed}_n{n}.json

%run scripts/openmax_eval.py

In [ ]:
# ── Optional: quick sanity-check OpenMax fit diagnostics ─────────────────
# Reload the OpenMax checkpoint and inspect its fitted Weibull params.

from python.src.models.openmax_trinet import OpenMaxTriNet
import torch

ckpt = PROJECT_ROOT / 'artifacts/checkpoints' / f'openmax_trinet_seed{SEED}_n{N_PER_CLASS}.pt'
if ckpt.exists():
    m = OpenMaxTriNet(num_classes=10, use_pretrained=False)
    m.load_state_dict(torch.load(ckpt, map_location='cpu'), strict=False)

    print(f'fitted          : {bool(m._fitted.item())}')
    print(f'threshold       : {float(m.threshold):.4f}')
    print(f'samples / class : {m.n_fit_per_class.tolist()}')
    print(f'Weibull shape   : {[round(v, 3) for v in m.weibull_shape.tolist()]}')
    print(f'Weibull scale   : {[round(v, 3) for v in m.weibull_scale.tolist()]}')
else:
    print('OpenMax checkpoint not found yet. Run Step 4 first.')